# 레슨 02 — URL 파라미터와 페이지네이션

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kevin-innovation/jupyter-lecture/blob/main/python-web-automation/lectures/02/%5B%ED%95%99%EC%83%9D%EC%9A%A9%5D%20%EB%A0%88%EC%8A%A8%2002%20%E2%80%94%20URL%20%ED%8C%8C%EB%9D%BC%EB%AF%B8%ED%84%B0%EC%99%80%20%ED%8E%98%EC%9D%B4%EC%A7%80%EB%84%A4%EC%9D%B4%EC%85%98.ipynb)

이 노트북은 읽기와 따라하기용 강의 노트북이다. 학생은 셀을 위에서 아래로 실행하며 입력 데이터가 어떤 구조로 바뀌는지 확인한다. URL 파라미터와 페이지네이션는 실제 업무 자동화에서 자주 등장하는 반복 패턴을 합성 fixture로 안전하게 연습한다.

## 학습 목표

1. URL을 path와 query string으로 분해한다.
2. urlencode로 안전한 검색 URL을 만든다.
3. 페이지 번호 규칙을 함수로 만든다.
4. pagination 링크에서 다음 페이지를 찾는다.
5. 여러 페이지 결과를 CSV로 저장한다.

---

## 1. URL을 구조로 읽기

URL은 긴 문자열이 아니라 scheme, domain, path, query로 나뉜 구조다.


In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, urlencode

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/02/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_bytes(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        return response.content
    return Path(DATA_BASE, filename).read_bytes()

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))


print('colab:', IS_COLAB)
print('data base:', DATA_BASE)

sample_url = 'https://example.com/library/search?q=python&category=all&page=2'
parsed = urlparse(sample_url)
print(parsed.path, parse_qs(parsed.query))



---

## 2. query string 만들기

검색 조건은 딕셔너리로 관리한 뒤 urlencode로 URL에 붙인다.


In [ ]:
base_url = 'https://example.com/library/search'
query = {'q': 'python automation', 'category': 'course', 'page': 1}
print(base_url + '?' + urlencode(query))



---

## 3. 검색 결과 카드

반복 단위는 article.result-card다. 카드 안에서 title, views, href를 읽는다.

~~~html
<article class="result-card" data-category="python" data-page="1"><h2 class="title"><a href="/library/python-1">자료</a></h2><span class="views">조회 1,240</span></article>
~~~


In [ ]:
html_text = load_text('search_page_1.html')
soup = BeautifulSoup(html_text, 'html.parser')
cards = soup.select('article.result-card')
print(len(cards), cards[0].select_one('.title a').text.strip())



---

## 4. 페이지네이션 순회

파일명 규칙을 함수로 만들면 페이지 수가 늘어나도 반복문만 바꾸면 된다.


In [ ]:
def page_filename(page):
    return f'search_page_{page}.html'
all_titles = []
for page in range(1, 4):
    page_soup = BeautifulSoup(load_text(page_filename(page)), 'html.parser')
    all_titles += [card.select_one('.title a').text.strip() for card in page_soup.select('article.result-card')]
print(len(all_titles))



---

## 데이터 출처와 안전 규칙

이 레슨의 파일은 모두 수업용 합성 데이터다. 실제 사이트의 개인정보, 로그인 정보, 유료 콘텐츠를 포함하지 않는다. 실제 사이트로 확장할 때는 robots.txt, 이용 약관, 요청 간격, 수집 목적을 먼저 확인한다. 수업 중에는 fixture를 반복 실행하며 구조를 익히고, 외부 사이트를 빠르게 반복 요청하지 않는다.

---

## 강의 보강 노트

이 절은 수업 중 교사가 질문으로 풀어낼 수 있는 운영형 설명이다. 학생이 셀을 실행한 뒤 결과만 맞히지 않고 자동화 절차를 말로 설명하도록 돕는다.

### 1. URL을 데이터 구조로 보기

검색어, 페이지 번호, 정렬 조건은 화면의 버튼처럼 보이지만 실제로는 query string에 들어간 데이터다. 학생에게 URL을 문자열 하나로 외우게 하지 말고, base URL과 params 딕셔너리로 나누어 읽게 하면 페이지네이션 코드가 훨씬 안정된다.

### 2. 페이지네이션 반복 범위

여러 페이지를 돌 때 가장 흔한 실수는 마지막 페이지를 임의로 정하거나 빈 페이지를 계속 요청하는 것이다. fixture에서는 1~3페이지가 제공되지만, 운영 코드에서는 다음 링크 존재 여부나 결과 개수 감소를 기준으로 멈추는 전략을 함께 설명한다.

### 3. 상대 링크 처리

목록 페이지에서 상세 링크가 `/item/3`처럼 상대 경로로 들어오는 경우가 많다. `urljoin`으로 절대 URL을 만든다는 원칙을 잡아두면 이후 다운로드, 이미지, 첨부파일 수집에서도 같은 사고방식을 재사용할 수 있다.

### 4. 검색 조건 로그

자동화가 어떤 조건으로 실행되었는지 남기지 않으면 같은 결과를 재현하기 어렵다. 검색어, page, sort, category 같은 파라미터는 결과 CSV와 별도의 실행 메모에 남기도록 지도한다.

### 5. 중복 결과 처리

페이지가 바뀌어도 같은 항목이 다시 나오는 상황은 실제 서비스에서도 발생한다. 학생 답안은 단순히 리스트를 이어 붙이는 것보다 item id나 URL을 key로 삼아 중복을 제거하는 방향으로 피드백한다.

### 6. 요청 안전성

URL 파라미터 연습은 실제 사이트에 무작정 요청을 보내기 쉬운 단원이다. 수업에서는 fixture를 사용하고, 실제 서비스로 확장할 때는 요청 간격과 robots 정책 확인이 먼저라는 점을 분명히 한다.

### 7. 디버깅 순서

결과가 비어 있으면 selector부터 바꾸기보다 먼저 조합된 URL, status code, HTML 일부, 선택된 card 개수를 차례대로 확인한다. 이 순서를 습관화하면 학생이 무작위 수정으로 시간을 잃지 않는다.

### 체크포인트 1: 오류 메시지 비교한다

수업 중 피드백 시간을 줄이기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 오류 메시지을/를 비교한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 2: 결과 행 수 요약한다

학생이 막힌 지점을 찾기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 결과 행 수을/를 요약한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 3: 검증 기준 검토한다

운영자가 결과를 이해할 수 있게, 레슨 02 URL 파라미터와 페이지네이션에서는 검증 기준을/를 검토한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 4: 중간 변수 정리한다

재실행했을 때 같은 결과를 얻기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 중간 변수을/를 정리한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 5: 입력 파일 설명한다

다음 셀에서 재사용하기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 입력 파일을/를 설명한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 6: 선택자 저장한다

실제 사이트 확장 전에 위험을 낮추기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 선택자을/를 저장한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 7: 오류 메시지 되돌아본다

저장 파일의 신뢰도를 높이기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 오류 메시지을/를 되돌아본다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 8: 결과 행 수 표준화한다

코랩과 로컬 실행 차이를 줄이기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 결과 행 수을/를 표준화한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 9: 검증 기준 검증한다

수업 중 피드백 시간을 줄이기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 검증 기준을/를 검증한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 10: 중간 변수 확인한다

학생이 막힌 지점을 찾기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 중간 변수을/를 확인한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 11: 입력 파일 분리한다

운영자가 결과를 이해할 수 있게, 레슨 02 URL 파라미터와 페이지네이션에서는 입력 파일을/를 분리한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 12: 선택자 기록한다

재실행했을 때 같은 결과를 얻기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 선택자을/를 기록한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 13: 오류 메시지 비교한다

다음 셀에서 재사용하기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 오류 메시지을/를 비교한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 14: 결과 행 수 요약한다

실제 사이트 확장 전에 위험을 낮추기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 결과 행 수을/를 요약한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 15: 검증 기준 검토한다

저장 파일의 신뢰도를 높이기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 검증 기준을/를 검토한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 16: 중간 변수 정리한다

코랩과 로컬 실행 차이를 줄이기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 중간 변수을/를 정리한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 17: 입력 파일 설명한다

수업 중 피드백 시간을 줄이기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 입력 파일을/를 설명한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 18: 선택자 저장한다

학생이 막힌 지점을 찾기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 선택자을/를 저장한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 19: 오류 메시지 되돌아본다

운영자가 결과를 이해할 수 있게, 레슨 02 URL 파라미터와 페이지네이션에서는 오류 메시지을/를 되돌아본다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 20: 결과 행 수 표준화한다

재실행했을 때 같은 결과를 얻기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 결과 행 수을/를 표준화한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 21: 검증 기준 검증한다

다음 셀에서 재사용하기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 검증 기준을/를 검증한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 22: 중간 변수 확인한다

실제 사이트 확장 전에 위험을 낮추기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 중간 변수을/를 확인한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 23: 입력 파일 분리한다

저장 파일의 신뢰도를 높이기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 입력 파일을/를 분리한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 24: 선택자 기록한다

코랩과 로컬 실행 차이를 줄이기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 선택자을/를 기록한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 25: 오류 메시지 비교한다

수업 중 피드백 시간을 줄이기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 오류 메시지을/를 비교한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 26: 결과 행 수 요약한다

학생이 막힌 지점을 찾기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 결과 행 수을/를 요약한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 27: 검증 기준 검토한다

운영자가 결과를 이해할 수 있게, 레슨 02 URL 파라미터와 페이지네이션에서는 검증 기준을/를 검토한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 28: 중간 변수 정리한다

재실행했을 때 같은 결과를 얻기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 중간 변수을/를 정리한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 29: 입력 파일 설명한다

다음 셀에서 재사용하기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 입력 파일을/를 설명한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 30: 선택자 저장한다

실제 사이트 확장 전에 위험을 낮추기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 선택자을/를 저장한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 31: 오류 메시지 되돌아본다

저장 파일의 신뢰도를 높이기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 오류 메시지을/를 되돌아본다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.

### 체크포인트 32: 결과 행 수 표준화한다

코랩과 로컬 실행 차이를 줄이기 위해, 레슨 02 URL 파라미터와 페이지네이션에서는 결과 행 수을/를 표준화한다는 과정을 별도의 단계로 둔다. 이때 학생은 화면에 보이는 값 하나보다 어떤 입력에서 어떤 변환을 거쳐 결과가 나왔는지 설명해야 한다. 강사는 변수명, 출력 형태, 저장 여부를 함께 확인하며 다음 실행에서도 같은 결과가 나오는지 묻는다.


# 레슨 02 — 실습 문제 정답지

> 🔒 교사·관리자 전용. 학생에게 배포 금지.

URL 파라미터와 페이지네이션 실습 문제의 모범 답안이다. 출력값만 보지 말고 selector, 타입 변환, 저장 흐름을 같이 확인한다.

## 0. 환경 셀


In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, urlencode

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/02/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_bytes(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        return response.content
    return Path(DATA_BASE, filename).read_bytes()

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))


print('colab:', IS_COLAB)
print('data base:', DATA_BASE)



---

## 문제 1 정답 — 검색 URL 분해하기


In [ ]:
sample_url = 'https://example.com/library/search?q=python&category=all&page=2'
parsed = urlparse(sample_url)
params = parse_qs(parsed.query)
print(parsed.path)
print(params['q'][0], params['category'][0], params['page'][0])



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력을 먼저 안정적인 자료구조로 바꾼 뒤 필요한 값만 선택한다. URL, query string, 페이지 번호와 카드 selector가 서로 연결되어야 한다. 중간 변수의 길이와 타입이 맞아야 뒤의 필터링, 저장, 요약이 모두 맞는다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 파일 경로가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 HTML 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 2 정답 — 쿼리 문자열 만들기


In [ ]:
base_url = 'https://example.com/library/search'
query = {'q': 'python automation', 'category': 'course', 'page': 1}
url = base_url + '?' + urlencode(query)
print(url)



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력을 먼저 안정적인 자료구조로 바꾼 뒤 필요한 값만 선택한다. URL, query string, 페이지 번호와 카드 selector가 서로 연결되어야 한다. 중간 변수의 길이와 타입이 맞아야 뒤의 필터링, 저장, 요약이 모두 맞는다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 파일 경로가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 HTML 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 3 정답 — 첫 페이지 HTML 읽기


In [ ]:
html_text = load_text('search_page_1.html')
soup = BeautifulSoup(html_text, 'html.parser')
print(soup.select_one('h1').text.strip())



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력을 먼저 안정적인 자료구조로 바꾼 뒤 필요한 값만 선택한다. URL, query string, 페이지 번호와 카드 selector가 서로 연결되어야 한다. 중간 변수의 길이와 타입이 맞아야 뒤의 필터링, 저장, 요약이 모두 맞는다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 파일 경로가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 HTML 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 4 정답 — 결과 카드 개수 세기


In [ ]:
cards = soup.select('article.result-card')
print('cards:', len(cards))



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력을 먼저 안정적인 자료구조로 바꾼 뒤 필요한 값만 선택한다. URL, query string, 페이지 번호와 카드 selector가 서로 연결되어야 한다. 중간 변수의 길이와 타입이 맞아야 뒤의 필터링, 저장, 요약이 모두 맞는다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 파일 경로가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 HTML 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 5 정답 — 첫 카드 제목과 href 읽기


In [ ]:
first = cards[0]
title = first.select_one('.title a').text.strip()
href = first.select_one('.title a')['href']
print(title)
print(href)



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력을 먼저 안정적인 자료구조로 바꾼 뒤 필요한 값만 선택한다. URL, query string, 페이지 번호와 카드 selector가 서로 연결되어야 한다. 중간 변수의 길이와 타입이 맞아야 뒤의 필터링, 저장, 요약이 모두 맞는다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 파일 경로가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 HTML 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 6 정답 — 상대 URL을 절대 URL로 바꾸기


In [ ]:
absolute = urljoin('https://example.com', href)
print(absolute)



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력을 먼저 안정적인 자료구조로 바꾼 뒤 필요한 값만 선택한다. URL, query string, 페이지 번호와 카드 selector가 서로 연결되어야 한다. 중간 변수의 길이와 타입이 맞아야 뒤의 필터링, 저장, 요약이 모두 맞는다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 파일 경로가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 HTML 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 7 정답 — 카드 하나를 딕셔너리로 만들기


In [ ]:
item = {'title': first.select_one('.title a').text.strip(), 'category': first['data-category'], 'date': first.select_one('time')['datetime'], 'views': clean_int(first.select_one('.views').text), 'url': urljoin('https://example.com', first.select_one('.title a')['href'])}
print(item)



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력을 먼저 안정적인 자료구조로 바꾼 뒤 필요한 값만 선택한다. URL, query string, 페이지 번호와 카드 selector가 서로 연결되어야 한다. 중간 변수의 길이와 타입이 맞아야 뒤의 필터링, 저장, 요약이 모두 맞는다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 파일 경로가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 HTML 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 8 정답 — 한 페이지 결과 리스트 만들기


In [ ]:
results = []
for card in cards:
    results.append({'title': card.select_one('.title a').text.strip(), 'category': card['data-category'], 'page': int(card['data-page']), 'views': clean_int(card.select_one('.views').text)})
print(results[0])
print(len(results))



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력을 먼저 안정적인 자료구조로 바꾼 뒤 필요한 값만 선택한다. URL, query string, 페이지 번호와 카드 selector가 서로 연결되어야 한다. 중간 변수의 길이와 타입이 맞아야 뒤의 필터링, 저장, 요약이 모두 맞는다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 파일 경로가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 HTML 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 9 정답 — 페이지 번호로 파일명 만들기


In [ ]:
def page_filename(page):
    return f'search_page_{page}.html'
for page in [1, 2, 3]:
    print(page_filename(page))



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력을 먼저 안정적인 자료구조로 바꾼 뒤 필요한 값만 선택한다. URL, query string, 페이지 번호와 카드 selector가 서로 연결되어야 한다. 중간 변수의 길이와 타입이 맞아야 뒤의 필터링, 저장, 요약이 모두 맞는다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 파일 경로가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 HTML 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 10 정답 — 3페이지 전체 순회하기


In [ ]:
all_results = []
for page in range(1, 4):
    page_soup = BeautifulSoup(load_text(page_filename(page)), 'html.parser')
    for card in page_soup.select('article.result-card'):
        all_results.append(card.select_one('.title a').text.strip())
print(len(all_results))
print(all_results[-1])



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력을 먼저 안정적인 자료구조로 바꾼 뒤 필요한 값만 선택한다. URL, query string, 페이지 번호와 카드 selector가 서로 연결되어야 한다. 중간 변수의 길이와 타입이 맞아야 뒤의 필터링, 저장, 요약이 모두 맞는다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 파일 경로가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 HTML 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 11 정답 — 다음 페이지 링크 찾기


In [ ]:
next_href = soup.select_one('a.next')['href']
next_query = parse_qs(urlparse(next_href).query)
print(next_href)
print(next_query['page'][0])



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력을 먼저 안정적인 자료구조로 바꾼 뒤 필요한 값만 선택한다. URL, query string, 페이지 번호와 카드 selector가 서로 연결되어야 한다. 중간 변수의 길이와 타입이 맞아야 뒤의 필터링, 저장, 요약이 모두 맞는다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 파일 경로가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 HTML 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 12 정답 — 조회수 1000 이상 필터링


In [ ]:
rich = []
for page in range(1, 4):
    page_soup = BeautifulSoup(load_text(page_filename(page)), 'html.parser')
    for card in page_soup.select('article.result-card'):
        views = clean_int(card.select_one('.views').text)
        if views >= 1000:
            rich.append(card.select_one('.title a').text.strip())
print(rich)



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력을 먼저 안정적인 자료구조로 바꾼 뒤 필요한 값만 선택한다. URL, query string, 페이지 번호와 카드 selector가 서로 연결되어야 한다. 중간 변수의 길이와 타입이 맞아야 뒤의 필터링, 저장, 요약이 모두 맞는다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 파일 경로가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 HTML 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 13 정답 — 카테고리별 개수 세기


In [ ]:
counts = {}
for page in range(1, 4):
    page_soup = BeautifulSoup(load_text(page_filename(page)), 'html.parser')
    for card in page_soup.select('article.result-card'):
        key = card['data-category']
        counts[key] = counts.get(key, 0) + 1
print(counts)



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력을 먼저 안정적인 자료구조로 바꾼 뒤 필요한 값만 선택한다. URL, query string, 페이지 번호와 카드 selector가 서로 연결되어야 한다. 중간 변수의 길이와 타입이 맞아야 뒤의 필터링, 저장, 요약이 모두 맞는다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 파일 경로가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 HTML 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 14 정답 — targets CSV 읽기


In [ ]:
target_rows = list(csv.DictReader(load_text('search_targets.csv').splitlines()))
for row in target_rows:
    print(row['query'], row['max_pages'])



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력을 먼저 안정적인 자료구조로 바꾼 뒤 필요한 값만 선택한다. URL, query string, 페이지 번호와 카드 selector가 서로 연결되어야 한다. 중간 변수의 길이와 타입이 맞아야 뒤의 필터링, 저장, 요약이 모두 맞는다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 파일 경로가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 HTML 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

## 문제 15 정답 — 검색 결과 CSV 저장하기


In [ ]:
rows = []
for page in range(1, 4):
    page_soup = BeautifulSoup(load_text(page_filename(page)), 'html.parser')
    for card in page_soup.select('article.result-card'):
        rows.append({'title': card.select_one('.title a').text.strip(), 'category': card['data-category'], 'views': clean_int(card.select_one('.views').text)})
with open('lesson02_results.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['title', 'category', 'views'])
    writer.writeheader()
    writer.writerows(rows)
print('saved:', 'lesson02_results.csv', len(rows))



### 왜 이 코드가 정답인지

이 답안은 문제에서 요구한 입력을 먼저 안정적인 자료구조로 바꾼 뒤 필요한 값만 선택한다. URL, query string, 페이지 번호와 카드 selector가 서로 연결되어야 한다. 중간 변수의 길이와 타입이 맞아야 뒤의 필터링, 저장, 요약이 모두 맞는다.

### 채점 포인트

- 입력 파일을 환경 셀의 헬퍼로 읽어 코랩과 로컬에서 모두 동작하는가.
- 반복 단위와 selector 또는 파일 경로가 fixture 구조와 정확히 대응하는가.
- 숫자, 날짜, 상대 경로처럼 후처리가 필요한 값을 그대로 두지 않았는가.
- 출력 형태가 문제 요구사항과 일치하는가.

### 자주 보이는 오답

- 이전 문제 selector를 그대로 복사해 현재 HTML 구조와 맞지 않는다.
- 문자열 숫자를 변환하지 않고 비교한다.
- 결과는 비슷하지만 중간 변수명이 불분명해 최종 미션에서 재사용하기 어렵다.

---

### 보강 설명 1

레슨 02 정답 해설은 실행 결과보다 과정 추적이 중요하다. 입력 파일, 반복 단위, selector 또는 경로, 변환 규칙을 분리하면 오류 위치를 빠르게 찾을 수 있다.

### 보강 설명 2

수업 중에는 학생에게 완성 코드를 먼저 보여주지 말고 HTML 구조나 CSV 헤더를 읽게 한다. 구조를 말로 설명할 수 있으면 코드는 짧아진다.

### 보강 설명 3

운영형 자동화는 다음 주에도 다시 실행되어야 한다. 그래서 파일명, URL, 저장 경로, 로그, 요약 문장을 코드 안에서 일관되게 남긴다.

### 보강 설명 4

실제 사이트로 확장할 때는 약관, robots.txt, 요청 간격, 개인정보 여부를 먼저 확인한다. 이 코스의 초반 fixture는 그 안전 습관을 만들기 위한 장치다.

### 보강 설명 5

레슨 02 정답 해설은 실행 결과보다 과정 추적이 중요하다. 입력 파일, 반복 단위, selector 또는 경로, 변환 규칙을 분리하면 오류 위치를 빠르게 찾을 수 있다.

### 보강 설명 6

수업 중에는 학생에게 완성 코드를 먼저 보여주지 말고 HTML 구조나 CSV 헤더를 읽게 한다. 구조를 말로 설명할 수 있으면 코드는 짧아진다.

### 보강 설명 7

운영형 자동화는 다음 주에도 다시 실행되어야 한다. 그래서 파일명, URL, 저장 경로, 로그, 요약 문장을 코드 안에서 일관되게 남긴다.

### 보강 설명 8

실제 사이트로 확장할 때는 약관, robots.txt, 요청 간격, 개인정보 여부를 먼저 확인한다. 이 코스의 초반 fixture는 그 안전 습관을 만들기 위한 장치다.

### 보강 설명 9

레슨 02 정답 해설은 실행 결과보다 과정 추적이 중요하다. 입력 파일, 반복 단위, selector 또는 경로, 변환 규칙을 분리하면 오류 위치를 빠르게 찾을 수 있다.

### 보강 설명 10

수업 중에는 학생에게 완성 코드를 먼저 보여주지 말고 HTML 구조나 CSV 헤더를 읽게 한다. 구조를 말로 설명할 수 있으면 코드는 짧아진다.

### 보강 설명 11

운영형 자동화는 다음 주에도 다시 실행되어야 한다. 그래서 파일명, URL, 저장 경로, 로그, 요약 문장을 코드 안에서 일관되게 남긴다.

### 보강 설명 12

실제 사이트로 확장할 때는 약관, robots.txt, 요청 간격, 개인정보 여부를 먼저 확인한다. 이 코스의 초반 fixture는 그 안전 습관을 만들기 위한 장치다.

### 보강 설명 13

레슨 02 정답 해설은 실행 결과보다 과정 추적이 중요하다. 입력 파일, 반복 단위, selector 또는 경로, 변환 규칙을 분리하면 오류 위치를 빠르게 찾을 수 있다.

### 보강 설명 14

수업 중에는 학생에게 완성 코드를 먼저 보여주지 말고 HTML 구조나 CSV 헤더를 읽게 한다. 구조를 말로 설명할 수 있으면 코드는 짧아진다.

### 보강 설명 15

운영형 자동화는 다음 주에도 다시 실행되어야 한다. 그래서 파일명, URL, 저장 경로, 로그, 요약 문장을 코드 안에서 일관되게 남긴다.


# 레슨 02 — 최종 미션 모범 답안

> 🔒 교사용. 학생에게는 최종 미션 문제 파일만 공유한다.

3페이지로 나뉜 자료실 검색 결과를 모두 수집해 CSV 리포트를 만든다.

## 0. 환경 셀


In [ ]:
import os
import re
import time
import csv
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, urlencode

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

IS_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ
if IS_COLAB:
    DATA_BASE = 'https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-web-automation/lectures/02/data'
else:
    DATA_BASE = './data'

def load_text(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        response.encoding = response.encoding or 'utf-8'
        return response.text
    return Path(DATA_BASE, filename).read_text(encoding='utf-8')

def load_bytes(filename):
    if DATA_BASE.startswith('http'):
        url = f'{DATA_BASE}/{filename}'
        response = requests.get(url, timeout=10, headers={'User-Agent': 'D-Lab-Lesson/1.0'})
        response.raise_for_status()
        return response.content
    return Path(DATA_BASE, filename).read_bytes()

def clean_int(text):
    return int(re.sub(r'[^0-9]', '', str(text)))


print('colab:', IS_COLAB)
print('data base:', DATA_BASE)



## 모범 답안


In [ ]:
def page_filename(page):
    return f'search_page_{page}.html'
rows = []
for page in range(1, 4):
    soup = BeautifulSoup(load_text(page_filename(page)), 'html.parser')
    for card in soup.select('article.result-card'):
        rows.append({'title': card.select_one('.title a').text.strip(), 'category': card['data-category'], 'views': clean_int(card.select_one('.views').text), 'url': urljoin('https://example.com', card.select_one('.title a')['href'])})
with open('lesson02_final_results.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['title', 'category', 'views', 'url'])
    writer.writeheader(); writer.writerows(rows)
print('rows:', len(rows))
print('high_view:', len([row for row in rows if row['views'] >= 1000]))



## 채점 메모

- 코드가 한 번 실행되어 산출물을 만들고, 다시 실행해도 같은 결과가 나와야 한다.
- 결과 저장 경로, 수집 개수, 필터 조건이 요약 문장과 충돌하지 않아야 한다.
- 실제 사이트로 옮길 때 필요한 요청 간격과 오류 처리 언급이 있어야 한다.

### 보강 설명 1

최종 미션 정답은 실행 결과보다 과정 추적이 중요하다. 입력 파일, 반복 단위, selector 또는 경로, 변환 규칙을 분리하면 오류 위치를 빠르게 찾을 수 있다.

### 보강 설명 2

수업 중에는 학생에게 완성 코드를 먼저 보여주지 말고 HTML 구조나 CSV 헤더를 읽게 한다. 구조를 말로 설명할 수 있으면 코드는 짧아진다.

### 보강 설명 3

운영형 자동화는 다음 주에도 다시 실행되어야 한다. 그래서 파일명, URL, 저장 경로, 로그, 요약 문장을 코드 안에서 일관되게 남긴다.

### 보강 설명 4

실제 사이트로 확장할 때는 약관, robots.txt, 요청 간격, 개인정보 여부를 먼저 확인한다. 이 코스의 초반 fixture는 그 안전 습관을 만들기 위한 장치다.

### 보강 설명 5

최종 미션 정답은 실행 결과보다 과정 추적이 중요하다. 입력 파일, 반복 단위, selector 또는 경로, 변환 규칙을 분리하면 오류 위치를 빠르게 찾을 수 있다.

### 보강 설명 6

수업 중에는 학생에게 완성 코드를 먼저 보여주지 말고 HTML 구조나 CSV 헤더를 읽게 한다. 구조를 말로 설명할 수 있으면 코드는 짧아진다.

### 보강 설명 7

운영형 자동화는 다음 주에도 다시 실행되어야 한다. 그래서 파일명, URL, 저장 경로, 로그, 요약 문장을 코드 안에서 일관되게 남긴다.

### 보강 설명 8

실제 사이트로 확장할 때는 약관, robots.txt, 요청 간격, 개인정보 여부를 먼저 확인한다. 이 코스의 초반 fixture는 그 안전 습관을 만들기 위한 장치다.

### 보강 설명 9

최종 미션 정답은 실행 결과보다 과정 추적이 중요하다. 입력 파일, 반복 단위, selector 또는 경로, 변환 규칙을 분리하면 오류 위치를 빠르게 찾을 수 있다.

### 보강 설명 10

수업 중에는 학생에게 완성 코드를 먼저 보여주지 말고 HTML 구조나 CSV 헤더를 읽게 한다. 구조를 말로 설명할 수 있으면 코드는 짧아진다.

### 보강 설명 11

운영형 자동화는 다음 주에도 다시 실행되어야 한다. 그래서 파일명, URL, 저장 경로, 로그, 요약 문장을 코드 안에서 일관되게 남긴다.

### 보강 설명 12

실제 사이트로 확장할 때는 약관, robots.txt, 요청 간격, 개인정보 여부를 먼저 확인한다. 이 코스의 초반 fixture는 그 안전 습관을 만들기 위한 장치다.


# 레슨 02 — 교사 가이드

## 학습 목표 (교사용)

- URL을 path와 query string으로 분해한다.
- urlencode로 안전한 검색 URL을 만든다.
- 페이지 번호 규칙을 함수로 만든다.
- pagination 링크에서 다음 페이지를 찾는다.
- 여러 페이지 결과를 CSV로 저장한다.

## 2시간 수업 흐름

| 시간 | 운영 | 확인 포인트 |
|---:|---|---|
| 0-15분 | 데이터 구조 읽기 | 반복 단위와 주요 속성 확인 |
| 15-45분 | 강의 예제 실행 | 환경 셀과 파일 로드 확인 |
| 45-85분 | 문제 1~10 풀이 | selector, 경로, 타입 변환 점검 |
| 85-110분 | 문제 11~15 풀이 | 집계와 CSV 저장 확인 |
| 110-120분 | 최종 미션 정리 | 산출물과 요약 문장 검수 |

## 사전 준비

- 코랩 링크가 Kevin-innovation/jupyter-lecture 저장소를 가리키는지 확인한다.
- data 폴더의 fixture 파일을 먼저 열어 학생이 볼 태그와 컬럼을 확인한다.
- 실제 사이트를 바로 요청하지 말고 합성 fixture로 구조를 읽게 한다.

## 질문 유도

- 반복 단위는 어떤 태그 또는 행인가?
- 화면에 보이는 텍스트와 속성 중 어느 값이 더 안정적인가?
- 숫자 비교 전에 어떤 변환이 필요한가?
- 이 자동화를 다음 주에도 실행한다면 어떤 로그가 필요한가?

## 채점 기준

15문제 중 12문제 이상 통과를 기본 완료로 본다. 최종 미션은 산출물 파일과 3문장 요약이 함께 있어야 완료 처리한다. 정답 코드와 다른 방식이어도 구조, 타입, 출력 형태가 맞으면 인정한다.

## 자주 발생하는 오류

| 오류 | 원인 | 지도 방법 |
|---|---|---|
| NoneType 오류 | selector가 틀림 | HTML에서 class/id를 다시 찾게 한다 |
| 파일 경로 오류 | 코랩과 로컬 경로 혼동 | DATA_BASE 출력부터 확인한다 |
| 숫자 비교 오류 | 문자열을 숫자로 변환하지 않음 | clean_int 또는 타입 변환 위치를 확인한다 |
| 결과 중복 | 반복 범위가 잘못됨 | 원본 item 수와 결과 리스트 수를 비교한다 |

## 확장 과제

결과를 CSV와 JSON 두 가지로 저장하거나, 처리 로그를 별도 리스트로 남기게 한다. 빠른 학생은 함수 분리와 오류 처리까지 진행한다.

### 보강 설명 1

교사 가이드은 실행 결과보다 과정 추적이 중요하다. 입력 파일, 반복 단위, selector 또는 경로, 변환 규칙을 분리하면 오류 위치를 빠르게 찾을 수 있다.

### 보강 설명 2

수업 중에는 학생에게 완성 코드를 먼저 보여주지 말고 HTML 구조나 CSV 헤더를 읽게 한다. 구조를 말로 설명할 수 있으면 코드는 짧아진다.

### 보강 설명 3

운영형 자동화는 다음 주에도 다시 실행되어야 한다. 그래서 파일명, URL, 저장 경로, 로그, 요약 문장을 코드 안에서 일관되게 남긴다.

### 보강 설명 4

실제 사이트로 확장할 때는 약관, robots.txt, 요청 간격, 개인정보 여부를 먼저 확인한다. 이 코스의 초반 fixture는 그 안전 습관을 만들기 위한 장치다.

### 보강 설명 5

교사 가이드은 실행 결과보다 과정 추적이 중요하다. 입력 파일, 반복 단위, selector 또는 경로, 변환 규칙을 분리하면 오류 위치를 빠르게 찾을 수 있다.

### 보강 설명 6

수업 중에는 학생에게 완성 코드를 먼저 보여주지 말고 HTML 구조나 CSV 헤더를 읽게 한다. 구조를 말로 설명할 수 있으면 코드는 짧아진다.

### 보강 설명 7

운영형 자동화는 다음 주에도 다시 실행되어야 한다. 그래서 파일명, URL, 저장 경로, 로그, 요약 문장을 코드 안에서 일관되게 남긴다.

### 보강 설명 8

실제 사이트로 확장할 때는 약관, robots.txt, 요청 간격, 개인정보 여부를 먼저 확인한다. 이 코스의 초반 fixture는 그 안전 습관을 만들기 위한 장치다.

### 보강 설명 9

교사 가이드은 실행 결과보다 과정 추적이 중요하다. 입력 파일, 반복 단위, selector 또는 경로, 변환 규칙을 분리하면 오류 위치를 빠르게 찾을 수 있다.

### 보강 설명 10

수업 중에는 학생에게 완성 코드를 먼저 보여주지 말고 HTML 구조나 CSV 헤더를 읽게 한다. 구조를 말로 설명할 수 있으면 코드는 짧아진다.

### 보강 설명 11

운영형 자동화는 다음 주에도 다시 실행되어야 한다. 그래서 파일명, URL, 저장 경로, 로그, 요약 문장을 코드 안에서 일관되게 남긴다.

### 보강 설명 12

실제 사이트로 확장할 때는 약관, robots.txt, 요청 간격, 개인정보 여부를 먼저 확인한다. 이 코스의 초반 fixture는 그 안전 습관을 만들기 위한 장치다.

### 보강 설명 13

교사 가이드은 실행 결과보다 과정 추적이 중요하다. 입력 파일, 반복 단위, selector 또는 경로, 변환 규칙을 분리하면 오류 위치를 빠르게 찾을 수 있다.

### 보강 설명 14

수업 중에는 학생에게 완성 코드를 먼저 보여주지 말고 HTML 구조나 CSV 헤더를 읽게 한다. 구조를 말로 설명할 수 있으면 코드는 짧아진다.
